# 🦀 Day 5: `Rc<RefCell<T>>` Pattern
---

In [ ]:
use std::rc::Rc;
use std::cell::RefCell;

// Rc<RefCell<T>> = multiple owners + mutable data
#[derive(Debug)]
struct SharedCounter {
    value: Rc<RefCell<i32>>,
}

impl SharedCounter {
    fn new(initial: i32) -> Self {
        SharedCounter { value: Rc::new(RefCell::new(initial)) }
    }
    fn clone_counter(&self) -> Self {
        SharedCounter { value: Rc::clone(&self.value) }
    }
    fn increment(&self) { *self.value.borrow_mut() += 1; }
    fn get(&self) -> i32 { *self.value.borrow() }
}

let counter = SharedCounter::new(0);
let c2 = counter.clone_counter();
let c3 = counter.clone_counter();

counter.increment();
c2.increment();
c3.increment();

println!("All see same value: {}", counter.get()); // 3

In [ ]:
// Practical: Observer pattern with shared mutable state
type Listener = Box<dyn Fn(i32)>;

struct Observable {
    value: i32,
    listeners: Vec<Listener>,
}

impl Observable {
    fn new(value: i32) -> Self { Observable { value, listeners: Vec::new() } }
    fn on_change(&mut self, f: impl Fn(i32) + 'static) {
        self.listeners.push(Box::new(f));
    }
    fn set(&mut self, new_value: i32) {
        self.value = new_value;
        for listener in &self.listeners {
            listener(new_value);
        }
    }
}

let log = Rc::new(RefCell::new(Vec::<String>::new()));
let log_clone = Rc::clone(&log);

let mut obs = Observable::new(0);
obs.on_change(move |v| { log_clone.borrow_mut().push(format!("Changed to {}", v)); });

obs.set(42);
obs.set(100);
println!("Log: {:?}", log.borrow());

## 🎯 Key Takeaways
1. `Rc<RefCell<T>>` = **shared ownership** + **interior mutability**
2. `Rc` handles ownership, `RefCell` handles mutation
3. Common in tree/graph structures and observer patterns
4. Borrow rules checked at runtime — can still panic!
5. Single-threaded only — use `Arc<Mutex<T>>` for threads

---
**Tomorrow:** `Cow<T>` — clone on write! 🐄